# Experimento — KAN, fase 2: dados e diagnóstico

Parte do campeão salvo na fase 1. Cada variação de dado muda apenas uma escolha, e sua comparação usa as mesmas cinco dobras. As transformações e o oversampling são ajustados somente dentro do treino da dobra.

Não acesse o teste nesta fase. As features candidatas (`tenure_1`, `cliente_novo`, `delta_preco`) já existem em `comum.py` e são usadas como hipótese, não como resultado garantido.

## 0. Ambiente e campeão da fase 1

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

CSV_PATH = os.environ.get("CHURN_CSV", "")
if CSV_PATH:
    os.environ["CHURN_CSV"] = CSV_PATH
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório churn_project.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import comum
from experimentos.kan import modelo as K

RESULTADOS = Path(comum.RESULTADOS)
PASTA_KAN = RESULTADOS / "kan"
PASTA_KAN.mkdir(parents=True, exist_ok=True)
print(f"Repositório: {ROOT}")
print(f"Dispositivo: {K.DEVICE}")
print(f"CSV: {comum.CAMINHO_CSV}")

In [ ]:
ARQUIVO = PASTA_KAN / "kan_fase1.json"
if not ARQUIVO.is_file():
    raise FileNotFoundError(f"Execute antes a fase anterior: {ARQUIVO}")
REGISTRO = json.loads(ARQUIVO.read_text(encoding="utf-8"))
CAMPEAO = dict(REGISTRO["config"])
CAMPEAO["hidden"] = tuple(CAMPEAO["hidden"])
FEATURES = tuple(REGISTRO.get("features", []))
OVER = REGISTRO.get("over", True)
print("Configuração recebida:", CAMPEAO)
print("Features:", FEATURES, "| oversampling:", OVER)

MODO_RAPIDO = False
SEMENTES = (42,) if MODO_RAPIDO else (42, 142, 242)

## 1. Blocos de dados

In [ ]:
estado = {"features": list(FEATURES), "over": OVER}
provas = [
    ("sem oversampling", {"over": False}),
    ("oversampling parcial", {"over": 0.5}),
    ("tenure_1", {"feature": "tenure_1"}),
    ("cliente_novo", {"feature": "cliente_novo"}),
    ("delta_preco", {"feature": "delta_preco"}),
]
if MODO_RAPIDO:
    provas = provas[:2]

resultados = []
for nome, mudanca in provas:
    anterior = {"features": list(estado["features"]), "over": estado["over"]}
    antes = K.medir_config(CAMPEAO, K.carregar_dobras(anterior["features"], anterior["over"]), SEMENTES)
    candidato = {"features": list(estado["features"]), "over": estado["over"]}
    if "over" in mudanca:
        candidato["over"] = mudanca["over"]
    else:
        candidato["features"].append(mudanca["feature"])
    depois = K.medir_config(CAMPEAO, K.carregar_dobras(candidato["features"], candidato["over"]), SEMENTES)
    delta = np.asarray(depois["ks_dobras"]) - np.asarray(antes["ks_dobras"])
    aceita = bool(delta.mean() > 0 and not MODO_RAPIDO)
    resultados.append({
        "variante": nome, "ks": depois["ks"], "delta_ks": float(delta.mean()),
        "ganhou_dobras": int((delta > 0).sum()), "n_dobras": len(delta),
        "features": candidato["features"], "over": candidato["over"],
        "anterior": anterior,
        "decisao": "aceita" if aceita else "em teste" if MODO_RAPIDO else "rejeitada",
    })
    if aceita:
        estado = candidato

display(pd.DataFrame(resultados))
print("Dados adotados:", estado)

## 2. Importância por permutação

In [ ]:
folds = K.carregar_dobras(estado["features"], estado["over"])
nomes = list(comum.carregar_df().drop(columns=["Churn"]).columns) + list(estado["features"])
importancias = []
for i, fold in enumerate(folds):
    Xtr, ytr, Xval, yval = fold
    model, _ = K.treinar(Xtr, ytr, CAMPEAO, 700 + i, Xval, yval)
    p0 = K.prever(model, Xval)
    rng = np.random.default_rng(700 + i)
    for j, nome in enumerate(nomes):
        Xperm = Xval.copy()
        Xperm[:, j] = rng.permutation(Xperm[:, j])
        queda = comum.ks(yval, p0) - comum.ks(yval, K.prever(model, Xperm))
        importancias.append({"feature": nome, "dobra": i, "queda_ks": queda})

imp = pd.DataFrame(importancias).groupby("feature").queda_ks.agg(["mean", "std"]).sort_values("mean", ascending=False)
display(imp)
ax = imp["mean"].sort_values().plot.barh(figsize=(8, 7), title="Queda média de KS ao permutar uma variável")
ax.set_xlabel("KS original - KS permutado")
plt.tight_layout()

## 3. Registro da decisão

In [ ]:
saida = {"config": {**CAMPEAO, "hidden": list(CAMPEAO["hidden"])},
         "features": estado["features"], "over": estado["over"],
         "resultados": resultados, "importancia_permutacao": imp.reset_index().to_dict("records"),
         "modo_rapido": MODO_RAPIDO}
arquivo_saida = PASTA_KAN / "kan_fase2.json"
arquivo_saida.write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding="utf-8")
if not MODO_RAPIDO:
    caminho_registro = ROOT / "experimentos" / "kan" / "preprocessamento" / "registro.json"
    registro = json.loads(caminho_registro.read_text(encoding="utf-8"))
    assinatura = comum.assinatura_teste()
    if assinatura != registro["assinatura_teste"]:
        raise ValueError("A assinatura do conjunto de teste mudou; não registre as validações.")
    pai = "v0"
    registro["atual"] = pai
    run_tag = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    for i, resultado in enumerate(resultados):
        anterior = resultado["anterior"]
        alteracoes = {}
        extras = [f for f in resultado["features"] if f not in anterior["features"]]
        if extras:
            alteracoes["extras"] = extras
        if resultado["over"] != anterior["over"]:
            alteracoes["oversampling"] = resultado["over"]
        slug = resultado["variante"].lower().replace(" ", "_").replace("=", "").replace(".", "_")
        ident = f"fase2_{run_tag}_{i + 1}_{slug}"
        versao = {
            "id": ident, "parte_de": pai, "mudanca": resultado["variante"],
            "motivo": "Teste pareado de uma estratégia de dados para a KAN.",
            "altera": alteracoes, "codigo": "experimentos/kan/kan_fase2.ipynb",
            "validacao": {
                "protocolo": f'5 dobras oficiais; {len(SEMENTES)} semente(s) de inicialização',
                "ks": resultado["ks"], "delta": resultado["delta_ks"],
                "ganhou": f'{resultado["ganhou_dobras"]}/{resultado["n_dobras"]}',
            },
            "decisao": resultado["decisao"], "evidencia": "resultados/kan/kan_fase2.json",
        }
        registro["versoes"].append(versao)
        if resultado["decisao"] == "aceita":
            pai = ident
            registro["atual"] = ident
    caminho_registro.write_text(json.dumps(registro, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Salvo em:", PASTA_KAN / "kan_fase2.json")